In [22]:
import pandas as pd
import sys
import numpy as np
import warnings
import os

sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

In [23]:
from estimation import cast_to_base_unit, calculate_contributions
from plots import plot_prediction
from retransform_prediction import retransform_
from retransform_data import retransform_data
from utils import cast_spec_to_dict, _convert_to_datetime

In [24]:
from dateutil.relativedelta import relativedelta

In [25]:
# !pip install shap
# import shap

In [26]:
def calculate_conf_bounds(pred, actual):
    # mape_series = np.abs((actual - pred) / actual).expanding(1).mean()
    # upper = (1+mape_series)*pred
    # lower = (1-mape_series)*pred
    std = (actual - pred).shift(1).expanding(2).std().fillna(0)
    low1, up1 = pred-std, pred+std
    low2, up2 = pred-3*std, pred+3*std
    return low1, low2, up1, up2

In [27]:
EPSILON = 1e-10

def rrse(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray=None):
    """ Root Relative Squared Error """
    return np.sqrt(
        np.sum(np.square(actual - predicted))
        / np.sum(np.square(actual - benchmark))
    )

def _error(actual: np.ndarray, predicted: np.ndarray):
    """ Simple error """
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """
    Percentage error

    Note: result is NOT multiplied by 100
    """
    return _error(actual, predicted) / (actual + EPSILON)

def mape(actual: np.ndarray, predicted: np.ndarray):
    """
    Mean Absolute Percentage Error

    Note: result is NOT multiplied by 100
    """
    return np.mean(np.abs(_percentage_error(actual, predicted)))

def mse(actual: np.ndarray, predicted: np.ndarray):
    """ Mean Squared Error """
    return np.mean(np.square(_error(actual, predicted)))


def rmse(actual: np.ndarray, predicted: np.ndarray):
    """ Root Mean Squared Error """
    return np.sqrt(mse(actual, predicted))


In [28]:
# def assign_weights(s):
#     if (s['directional_accuracy'] == -1) and (s['within_cbounds'] == -1):
#         return 2
#     elif (s['directional_accuracy'] == -1) and (s['within_cbounds'] == 1):
#         return 1.75
#     elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == -1):
#         return 1.25
#     elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == 1):
#         return 1
#     else: return np.infty

# confidence_bounds_func = lambda row: row['lower']<=row['y_pred']<=row['upper']

# def wdmpe(predicted, actual):
#     actual_diff = actual.sort_index().diff()
#     actual_signs = np.sign(actual_diff)
#     predicted_diff = predicted.sort_index().diff()
#     predicted_signs = np.sign(predicted_diff)

#     resid = predicted-actual

#     dir_acc = list(actual_signs * predicted_signs)

#     resid_mean = resid.expanding(1).mean()
#     resid_std = resid.expanding(2).std().fillna(0)

#     lower = actual-resid_std
#     upper = actual+resid_std

#     df = pd.DataFrame({
#         "directional_accuracy": dir_acc,
#         "lower": lower,
#         "upper": upper,
#         "y_pred": predicted
#     }).iloc[1:, :]
#     df['within_cbounds'] = df.apply(confidence_bounds_func, axis=1).astype(int).replace({0: -1})
#     df['percentage_error'] = resid / actual

#     df['weights'] = df.apply(assign_weights, axis=1)
#     df["weighted_percentage_error"] = df['weights'] * df['percentage_error']
#     return df["weighted_percentage_error"].mean()


In [29]:
# def cast_to_base_unit(ds, model_result, spec, series_name):
#     Spec = cast_spec_to_dict(spec.loc[spec["seriesid"] == series_name])

#     ## Retransform
#     ds = _convert_to_datetime(ds, ['ReferenceDate'])

#     dsrc = ds.set_index('ReferenceDate')

#     # def retransform_prediction(transf_series, base_series, Spec, series_name):
#     base_series = dsrc[series_name]
#     header = [series_name]

#     backcast = model_result['predictions']['backcast']
#     forecast = pd.Series(
#         model_result["predictions"]["forecast"],
#         index=[model_result["predictions"]["reference_date"]]
#         )

#     transf_pred = pd.concat([backcast, forecast])
#     transf_pred.index = pd.to_datetime(transf_pred.index)

#     transf_series = model_result["actual"]

#     Time = np.sort(np.unique(np.concatenate((base_series.index.date, transf_pred.index.date))))
#     cutoff_date = transf_pred.index.min().date()

#     Z = base_series.reindex(Time).to_numpy().reshape(-1,1)

#     Yhat = transf_pred.reindex(Time).to_numpy().reshape(-1,1)
#     Y = transf_series.reindex(Time).to_numpy().reshape(-1,1)

#     Rhat = retransform_(X=Yhat, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)
#     R = retransform_data(X=Y, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)

#     return Rhat, R, Time, cutoff_date

In [30]:
from estimation import select_model_by_r2, ml_fit_predict
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lineartree import LinearForestRegressor, LinearBoostRegressor
from statsmodels.tsa.api import VAR

ref_date_col = "ReferenceDate"

def estimate_automl(
    ds, ds_base, spec, ref_date_col, series_name, reference_date, n_periods
):
    """
    Automatically trains models, evaluates them, and selects the best one based on R-squared.

    Parameters:
    - ds (pd.DataFrame): Dataset containing features and target.
    - ref_date_col (str): Column name for reference dates.
    - series_name (str): Column name for the series to forecast.
    - reference_date (datetime or str): Date for forecasting and backcasting split.
    - n_periods (int): Number of periods to forecast.

    Returns:
    - dict: A dictionary containing the best model, its R-squared score, and predictions.
    """
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(),
        "LinearForest": LinearForestRegressor(
            base_estimator=Ridge(), random_state=42, max_features="log2"
        ),
        "LinearBoost": LinearBoostRegressor(
            base_estimator=Ridge(), random_state=42, max_features="log2"
        ),
        "RandomForestRegressor": RandomForestRegressor()
    }

    models_results = {}

    for model_name, model in models.items():
        coef_, pred, T, values = ml_fit_predict(
            ds=ds,
            ref_date_col=ref_date_col,
            model=model,
            series_name=series_name,
            reference_date=reference_date,
            n_periods=n_periods,
        )

        models_results[model_name] = {
            "backcast": pred["y_pred"].drop(reference_date),
            "forecast": pred["y_pred"].loc[reference_date],
            "reference_date": reference_date,
            "coef_": coef_,
            "values": values
        }
    # Ensure all predictions align with the actuals index
    y_actual = ds.set_index(ref_date_col).loc[T].sort_index()[series_name]

    # Select the best model based on R-squared
    best_model_res= select_model_by_r2(
        models_results, y_actual.drop(reference_date)
    )
    best_model_res["actual"] = y_actual
    best_model_res["rmse"] = rmse(
        actual=y_actual.drop(reference_date),
        predicted=models_results[model_name]["backcast"],
    )
    best_model_res["mape"] = mape(
        actual=y_actual.drop(reference_date),
        predicted=models_results[model_name]["backcast"],
    )

    # Rhat, R, Time, cutoff_date = cast_to_base_unit(
    #     ds=ds_base, model_result=best_model_info, spec=spec, series_name=series_name
    # )
    # TBC

    return best_model_res

In [31]:
series_name = "PCEC96"
reference_date = "2024-08-01"
n_periods = 60
plt_out_dir = "../data/07_model_output/"

In [32]:
ds = pd.read_parquet("../data/04_feature/selected_series.parquet")

In [ ]:
ds_spec = pd.read_csv("../data/02_intermediate/variable.csv")
ds_base = pd.read_parquet("../data/02_intermediate/non_transformed_data.parquet")

# Example usage
best_model_result = estimate_automl(
    ds=ds,
    ds_base=ds_base,
    spec=ds_spec,
    ref_date_col="ReferenceDate",
    series_name=series_name,
    reference_date=reference_date,
    n_periods=n_periods,
)

# Print the best model's details
print("===== Best Model Details =====")
print(f"Model                     : {best_model_result['best_model']}")
print(f"Reference Date            : {reference_date}")
print(f"Forecast                  : {best_model_result['pred_']['forecast']:.4f}")
print(f"R-Squared (R²)            : {best_model_result['r_squared']:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {best_model_result['mape']:.2f}%")
print(f"Root Mean Square Error (RMSE) : {best_model_result['rmse']:.4f}")

formula = ds_spec.loc[ds_spec["seriesid"] == series_name]["transformation"].item()
unit = ds_spec.loc[ds_spec["seriesid"] == series_name]["units"].item()

dt = best_model_result['pred_']['backcast'].index
plot_prediction(
    dt=dt,
    y_pred=best_model_result['pred_']['backcast'],
    y_actual=best_model_result['actual'].loc[dt],
    mode="lines+markers",
    title=f"Series: {series_name}, Reference Date: {reference_date}, Unit: {unit} {formula}",
    )


Rhat, R, Time, cutoff_date = cast_to_base_unit(ds_base, best_model_result, ds_spec, series_name)

# TBC
header = [series_name]
Rhat_df = pd.DataFrame(Rhat, columns=header, index=Time)
R_df = pd.DataFrame(R, columns=header, index=Time)

reference_date = pd.to_datetime(reference_date).date()
retr_forecast = Rhat_df.loc[reference_date].item()
retr_actual = R_df.loc[reference_date].item()

print("\n===== Forecast vs Actual =====")
print(f"Reference Date            : {reference_date}")
print(f"Forecast                  : {best_model_result['pred_']['forecast']:.4f}")
print(f"Forecast (retransformed)  : {retr_forecast:,.2f}")
print(f"Actual Release            : {retr_actual:,.2f}")
print(f"Percentage Error (Level)  : {(retr_forecast - retr_actual) / retr_actual:.2%}")

Z_df = pd.DataFrame(ds[series_name], columns=header, index=Time)

pred = Rhat_df.loc[dt][series_name]
actual = R_df.loc[dt][series_name]
low1, low2, up1, up2 = calculate_conf_bounds(pred, actual)

plot_prediction(
    dt=dt,
    y_pred=pred,
    y_actual=actual,
    mode="lines+markers",
    title=f"Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}",
    )

plot_prediction(
    dt=dt,
    y_pred=Rhat_df.loc[dt][series_name],
    y_actual=R_df.loc[dt][series_name],
    mode="lines+markers",
    lower1=low1,
    upper1=up1,
    lower2=low2,
    upper2=up2,
    title=f"Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}",
    )

# # print(f"y_pred: {R_df.loc[dt][series_name].item()}, y_actual: {Z_df.loc[dt][series_name].item()}")